In [1]:
import os
import sys
module_path = os.path.join("./KeyPoint-Analysis/KPG/")
sys.path.insert(0, module_path)
from rouge_setbase import preprocess_dataset, compute_rouge, compute_rouge_max
from softF1 import softevaluation
root_dir_pth = "../"

import pandas as pd
# from openai import OpenAI
import re
import ast

2026-01-13 02:27:42.019795: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-13 02:27:42.041949: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-13 02:27:42.041973: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-13 02:27:42.041986: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-13 02:27:42.047193: I tensorflow/core/platform/cpu_feature_g

# Read Data

In [2]:
ground_truth_df = pd.read_pickle("../data/test/test.pkl")
ground_truth_df = ground_truth_df[['category', 'product_name', 'user_id', 'claim_split_gold']]
ground_truth_df = ground_truth_df.rename(columns={'claim_split_gold': 'key_point_given'})
ground_truth_df

,category,product_name,user_id,key_point_given
0,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[Head & Shoulders Normal Hair Shampoo contains...
1,Beauty,Gillette Mach 3 Razor,5000858,[Swapping blades is easy with the single-point...
2,Beauty,Pitrok,5296801,[PitRok is gentle and effective for sensitive ...
3,Beauty,Gillette Mach 3 Razor,5050855,[Replacement blades are more expensive compare...
4,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,[The shampoo gently enhances natural highlight...
...,...,...,...,...
95,Travel,Dollar Rent A Car Worldwide,5297771,[Dollar Rent A Car Worldwide offers consistent...
96,Travel,Amsterdam (Netherlands),5202501,[Accommodation options range from budget-frien...
97,Travel,Leicester in General,5020891,[Leicester offers a surprisingly rich mix of e...
98,Travel,Milan in general,5091015,[Public transport in Milan is generally excell...


In [3]:
root_path = f"../output/stage_2_rl_inference/summary_kp_extraction"
temp_df = pd.read_pickle(root_path + "/1/1_done.pkl")
temp_df = temp_df.rename(columns={'voter_full': 'user_id'})
temp_df.shape

(96, 9)

In [4]:
temp_df['claim_split_predicted'] = temp_df['claim_split_predicted'].apply(lambda x: x.strip("```json").strip("\n"))
temp_df['claim_split_predicted'] = temp_df['claim_split_predicted'].apply(lambda x: ast.literal_eval(x))

In [5]:
temp_df = temp_df.rename(columns={'claim_split_predicted': 'key_point'})

In [6]:
claim_split_predicted = temp_df.merge(ground_truth_df)

In [7]:
claim_split_predicted

,key_point,personalized_summaries,category,product_name,user_id,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category,key_point_given
0,[Head & Shoulders Normal Hair Shampoo is effec...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,[Head & Shoulders Normal Hair Shampoo contains...
1,[The Gillette Mach 3 Razor provides a close an...,Based on the user profile and the helpful key ...,Beauty,Gillette Mach 3 Razor,5000858,[ ~ ~ OK girls. This opinion is really for ...,[ When I had my red low lights done a coupl...,[ I bought these for going to festivals in ...,1,[Swapping blades is easy with the single-point...
2,[Pitrok is a natural deodorant free from artif...,Based on the helpful key points and the user p...,Beauty,Pitrok,5296801,[ this is the natural option to fight per...,[ 8776; Dandruff - the evil flaky whit...,[ 02 is I think one of the best innovations...,1,[PitRok is gentle and effective for sensitive ...
3,[The Gillette Mach 3 Razor delivers a close an...,Here is a personalized summary of product A (G...,Beauty,Gillette Mach 3 Razor,5050855,[ ~ ~ OK girls. This opinion is really for ...,[ They are the 23.30.From five days I don t...,[ i love alberto balsams esspecially the su...,1,[Replacement blades are more expensive compare...
4,"[The shampoo is gentle and effective., It is a...",Based on the user profile and the helpful key ...,Beauty,Timotei Golden Highlights Camomile Shampoo,5332164,[ Searching the freebie sites I found sache...,"[ Has a lovely light smell, really pretty a...","[ Has a lovely light smell, really pretty a...",1,[The shampoo gently enhances natural highlight...
...,...,...,...,...,...,...,...,...,...,...
91,[Dollar Rent A Car Worldwide is a reliable and...,Based on the helpful key points and user 111's...,Travel,Dollar Rent A Car Worldwide,5297771,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,[ AH THE DREAM.- The thrill of the open ro...,1,[Dollar Rent A Car Worldwide offers consistent...
92,[Amsterdam is steeped in tradition and is very...,Here is a personalized summary of product A (A...,Travel,Amsterdam (Netherlands),5202501,[ A week off of University and online trav...,[ Air Canada isn t that bad of an airline. ...,[ Air Canada isn t that bad of an airline. ...,1,[Accommodation options range from budget-frien...
93,[Leicester offers a diverse range of shopping ...,Here is a personalized summary of product A (L...,Travel,Leicester in General,5020891,"[ Having lived here for 25 years now, this ...",[ Visitors to London who get the London Pas...,[ Visitors to London who get the London Pas...,1,[Leicester offers a surprisingly rich mix of e...
94,[Milan features stunning architecture that imp...,Milan is a city that will leave you in awe. Fr...,Travel,Milan in general,5091015,[ Milculo is a swear word I was born near M...,[ Milan undoubtedly has some of the finest ...,[ Milan undoubtedly has some of the finest ...,1,[Public transport in Milan is generally excell...


In [8]:
merged_df = claim_split_predicted.explode(['key_point']).explode(['key_point_given'])

In [9]:
# merged_df[merged_df['product_name'].str.contains('Morrisons')]
merged_df

,key_point,personalized_summaries,category,product_name,user_id,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category,key_point_given
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,Head & Shoulders Normal Hair Shampoo contains ...
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,The shampoo keeps your scalp clear and flake-f...
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,Some users have found it doubles as a body was...
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,Head & Shoulders offers formulas tailored to d...
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,The shampoo is effective for dandruff control ...
...,...,...,...,...,...,...,...,...,...,...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,Classical concerts and opera performances are ...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,Gellert Bath House offers stunning architectur...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,Nightlife options like Bahnhof nightclub have ...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,"Budapest balances rich history, cultural delig..."


In [10]:
merged_df['key_point_given'] = merged_df['key_point_given'].apply(lambda x: x['key_point'] if type(x) == dict and 'key_point' in x else x)

# Evaluation

In [11]:
from softF1 import *

## BERTScore

### SoftPrecision

In [12]:
softp_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point'])\
    .apply(lambda grp: "".join([cand + "=" for cand in (grp['key_point_given'].tolist())])).reset_index(name='multi_cands')
softp_data

/tmp/ipykernel_2829/1500243889.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  softp_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point'])\


,category,product_name,user_id,key_point,multi_cands
0,Beauty,Gillette Mach 3 Razor,5000858,The Gillette Mach 3 Razor provides a close and...,Swapping blades is easy with the single-point ...
1,Beauty,Gillette Mach 3 Razor,5000858,The Mach 3 Razor is effective at providing a c...,Swapping blades is easy with the single-point ...
2,Beauty,Gillette Mach 3 Razor,5000858,The razor has a comfortable grip and a simple ...,Swapping blades is easy with the single-point ...
3,Beauty,Gillette Mach 3 Razor,5000858,The razor is a good value for the price.,Swapping blades is easy with the single-point ...
4,Beauty,Gillette Mach 3 Razor,5000858,The razor is a great choice for those with sen...,Swapping blades is easy with the single-point ...
...,...,...,...,...,...
825,Travel,Milan in general,5091015,Milan provides an authentic Italian dining exp...,Public transport in Milan is generally excelle...
826,Travel,Milan in general,5091015,Milan's vibrant shopping scene is a highlight ...,Public transport in Milan is generally excelle...
827,Travel,Milan in general,5091015,The city has a strong sense of community that ...,Public transport in Milan is generally excelle...
828,Travel,Milan in general,5091015,The city is a paradise for fashion lovers.,Public transport in Milan is generally excelle...


In [13]:
cands, refs= preprocess_text(softp_data, metrics = "softPrecision")

P, R, F = bert_scorer.score(cands, refs)
P_average = P.mean()

In [14]:
P_average

tensor(0.4787)

### SoftRecall

In [15]:
softr_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point_given'])\
    .apply(lambda grp: "".join([cand + "=" for cand in (grp['key_point'].tolist())])).reset_index(name='multi_cands')
softr_data

/tmp/ipykernel_2829/4013153830.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  softr_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point_given'])\


,category,product_name,user_id,key_point_given,multi_cands
0,Beauty,Gillette Mach 3 Razor,5000858,Swapping blades is easy with the single-point ...,The Gillette Mach 3 Razor provides a close and...
1,Beauty,Gillette Mach 3 Razor,5000858,The blades last longer than average disposable...,The Gillette Mach 3 Razor provides a close and...
2,Beauty,Gillette Mach 3 Razor,5000858,The grip is comfortable and secure.,The Gillette Mach 3 Razor provides a close and...
3,Beauty,Gillette Mach 3 Razor,5000858,The lubricating strip helps the razor glide sm...,The Gillette Mach 3 Razor provides a close and...
4,Beauty,Gillette Mach 3 Razor,5000858,The razor causes fewer nicks and less irritati...,The Gillette Mach 3 Razor provides a close and...
...,...,...,...,...,...
1266,Travel,Milan in general,5091015,The Monday morning market along the main canal...,Milan features stunning architecture that impr...
1267,Travel,Milan in general,5091015,The integrated transport system offers great v...,Milan features stunning architecture that impr...
1268,Travel,Milan in general,5091015,The market near Piazzale Piemonte and Piazza S...,Milan features stunning architecture that impr...
1269,Travel,Milan in general,5091015,Trams run clean and on time.,Milan features stunning architecture that impr...


In [16]:
cands, refs= preprocess_text(softr_data, metrics = "softRecall")

P, R, F = bert_scorer.score(cands, refs)
R_average = R.mean()

In [17]:
R_average

tensor(0.4413)

### F1

In [18]:
result = softF1(P_average, R_average)

In [19]:
result

0.4592415529132738

## BARTScore

### SoftPrecision

In [20]:
softp_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point'])\
    .apply(lambda grp: "".join([cand + "=" for cand in (grp['key_point_given'].tolist())])).reset_index(name='multi_cands')
softp_data

/tmp/ipykernel_2829/1500243889.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  softp_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point'])\


,category,product_name,user_id,key_point,multi_cands
0,Beauty,Gillette Mach 3 Razor,5000858,The Gillette Mach 3 Razor provides a close and...,Swapping blades is easy with the single-point ...
1,Beauty,Gillette Mach 3 Razor,5000858,The Mach 3 Razor is effective at providing a c...,Swapping blades is easy with the single-point ...
2,Beauty,Gillette Mach 3 Razor,5000858,The razor has a comfortable grip and a simple ...,Swapping blades is easy with the single-point ...
3,Beauty,Gillette Mach 3 Razor,5000858,The razor is a good value for the price.,Swapping blades is easy with the single-point ...
4,Beauty,Gillette Mach 3 Razor,5000858,The razor is a great choice for those with sen...,Swapping blades is easy with the single-point ...
...,...,...,...,...,...
825,Travel,Milan in general,5091015,Milan provides an authentic Italian dining exp...,Public transport in Milan is generally excelle...
826,Travel,Milan in general,5091015,Milan's vibrant shopping scene is a highlight ...,Public transport in Milan is generally excelle...
827,Travel,Milan in general,5091015,The city has a strong sense of community that ...,Public transport in Milan is generally excelle...
828,Travel,Milan in general,5091015,The city is a paradise for fashion lovers.,Public transport in Milan is generally excelle...


In [21]:
cands, refs= preprocess_text(softp_data, metrics = "softPrecision")

#BART score cannot be processed for different numbers of reference sentences, so
#check for the maximum number of reference sentences and match the size.
#We fill in the None for the missing sentences because we only pick one of the 
#maximum values and not the average, so there is no impact on performance

refs = balance_ref_num(refs)        

# generation scores from the first list of texts to the second list of texts.
P = bart_scorer.multi_ref_score(cands, refs, agg="max", batch_size=4) # agg means aggregation, can be mean or max

#mapping the score to (0,1]
P_average = math.tanh(math.exp((mean(P))/2+1.3))
#P_average = math.tanh(mean(P)) + 1
#P_average = math.exp(mean(P))

In [22]:
P_average

0.7189991656410629

### SoftRecall

In [23]:
softr_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point_given'])\
    .apply(lambda grp: "".join([cand + "=" for cand in (grp['key_point'].tolist())])).reset_index(name='multi_cands')
softr_data

/tmp/ipykernel_2829/4013153830.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  softr_data = merged_df.groupby(['category', 'product_name', 'user_id', 'key_point_given'])\


,category,product_name,user_id,key_point_given,multi_cands
0,Beauty,Gillette Mach 3 Razor,5000858,Swapping blades is easy with the single-point ...,The Gillette Mach 3 Razor provides a close and...
1,Beauty,Gillette Mach 3 Razor,5000858,The blades last longer than average disposable...,The Gillette Mach 3 Razor provides a close and...
2,Beauty,Gillette Mach 3 Razor,5000858,The grip is comfortable and secure.,The Gillette Mach 3 Razor provides a close and...
3,Beauty,Gillette Mach 3 Razor,5000858,The lubricating strip helps the razor glide sm...,The Gillette Mach 3 Razor provides a close and...
4,Beauty,Gillette Mach 3 Razor,5000858,The razor causes fewer nicks and less irritati...,The Gillette Mach 3 Razor provides a close and...
...,...,...,...,...,...
1266,Travel,Milan in general,5091015,The Monday morning market along the main canal...,Milan features stunning architecture that impr...
1267,Travel,Milan in general,5091015,The integrated transport system offers great v...,Milan features stunning architecture that impr...
1268,Travel,Milan in general,5091015,The market near Piazzale Piemonte and Piazza S...,Milan features stunning architecture that impr...
1269,Travel,Milan in general,5091015,Trams run clean and on time.,Milan features stunning architecture that impr...


In [24]:
cands, refs= preprocess_text(softr_data, metrics = "softRecall")

refs = balance_ref_num(refs)

R = bart_scorer.multi_ref_score(cands, refs, agg="max", batch_size=4)

#mapping the score to (0,1]
R_average = math.tanh(math.exp((mean(R)/2)+1.3)) 
#R_average = (0.25 * mean(R)) + 1
#R_average = math.exp(mean(R))

In [25]:
R_average

0.7727055131858012

### F1

In [26]:
result = softF1(P_average, R_average)

In [27]:
result

0.7448855355253915

## BLEURTScore

In [28]:
softp_data = merged_df.sort_values(by=['category', 'product_name', 'user_id', 'key_point'])
df_compare_precision = softp_data[['key_point', 'key_point_given']].rename(columns={'key_point': 'candidate', 'key_point_given': 'reference'})
df_compare_precision

,candidate,reference
1,The Gillette Mach 3 Razor provides a close and...,Swapping blades is easy with the single-point ...
1,The Gillette Mach 3 Razor provides a close and...,The razor's design prevents hair from clogging...
1,The Gillette Mach 3 Razor provides a close and...,The swivel head and spring mechanism allow the...
1,The Gillette Mach 3 Razor provides a close and...,The razor causes fewer nicks and less irritati...
1,The Gillette Mach 3 Razor provides a close and...,The three-blade system glides effortlessly for...
...,...,...
94,The iconic Duomo is a must-see attraction in M...,The market near Piazzale Piemonte and Piazza S...
94,The iconic Duomo is a must-see attraction in M...,Visiting during the San Ambrogio festival adds...
94,The iconic Duomo is a must-see attraction in M...,Favoring trams over buses during peak times im...
94,The iconic Duomo is a must-see attraction in M...,Choosing markets over touristy eateries provid...


In [29]:
candidates = df_compare_precision['candidate']
references = df_compare_precision['reference']

In [30]:
import tensorflow as tf

In [31]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


2026-01-13 02:30:56.290508: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:56.292814: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:56.292972: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [32]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 17202740986126958369
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 19436404736
locality {
  bus_id: 1
  links {
  }
}
incarnation: 1018452051459826277
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


2026-01-13 02:30:56.314106: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:56.314324: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:56.314439: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:56.314841: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:56.314994: I tensorflow/compile

In [33]:
# with tf.device('/gpu:0'):
#   result = calculatingScore(references, candidates)
#   df_compare_precision["BLEURT Score"] = result

#   #After calculating the semantic quality of all candidates and reference pairs, the one with the highest score is selected as the correct pair.
#   df_bestkp_pair_precision = df_compare_precision.loc[df_compare_precision.groupby(["candidate"])["BLEURT Score"].idxmax()]
#   #take average of all best scores as the soft precision score.
#   P_average = df_bestkp_pair_precision["BLEURT Score"].mean()

### SoftPrecision

In [34]:
softp_data = merged_df.sort_values(by=['category', 'product_name', 'user_id', 'key_point'])
df_compare_precision = softp_data[['key_point', 'key_point_given']].rename(columns={'key_point': 'candidate', 'key_point_given': 'reference'})
df_compare_precision

,candidate,reference
1,The Gillette Mach 3 Razor provides a close and...,Swapping blades is easy with the single-point ...
1,The Gillette Mach 3 Razor provides a close and...,The razor's design prevents hair from clogging...
1,The Gillette Mach 3 Razor provides a close and...,The swivel head and spring mechanism allow the...
1,The Gillette Mach 3 Razor provides a close and...,The razor causes fewer nicks and less irritati...
1,The Gillette Mach 3 Razor provides a close and...,The three-blade system glides effortlessly for...
...,...,...
94,The iconic Duomo is a must-see attraction in M...,The market near Piazzale Piemonte and Piazza S...
94,The iconic Duomo is a must-see attraction in M...,Visiting during the San Ambrogio festival adds...
94,The iconic Duomo is a must-see attraction in M...,Favoring trams over buses during peak times im...
94,The iconic Duomo is a must-see attraction in M...,Choosing markets over touristy eateries provid...


In [35]:
candidates = df_compare_precision['candidate']
references = df_compare_precision['reference']

In [36]:
import tensorflow as tf

In [37]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [38]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 443034048829707967
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 19436404736
locality {
  bus_id: 1
  links {
  }
}
incarnation: 14718263321694918086
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


2026-01-13 02:30:56.442023: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:56.442253: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:56.442368: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:56.442670: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:56.442679: I tensorflow/core/co

In [39]:
result = calculatingScore(references, candidates)
df_compare_precision["BLEURT Score"] = result

#After calculating the semantic quality of all candidates and reference pairs, the one with the highest score is selected as the correct pair.
df_bestkp_pair_precision = df_compare_precision.loc[df_compare_precision.groupby(["candidate"])["BLEURT Score"].idxmax()]
#take average of all best scores as the soft precision score.
P_average = df_bestkp_pair_precision["BLEURT Score"].mean()

INFO:tensorflow:Reading checkpoint /mnt/d/Desktop/PHD READING/HELPFULSUMM/evaluation/KeyPoint-Analysis/KPG/BLEURT-20.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint BLEURT-20
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:BLEURT-20
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... max_seq_length:512
INFO:tensorflow:... vocab_file:None
INFO:tensorflow:... do_lower_case:None
INFO:tensorflow:... sp_model:sent_piece
INFO:tensorflow:... dynamic_seq_length:True
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Will load model: /mnt/d/Desktop/PHD READING/HELPFULSUMM/evaluation/KeyPoint-Analysis/KPG/BLEURT-20/sent_piece.model.
INFO:tensorflow:SentencePiece tokenizer created.
INFO:tensorflow:Creating Eager Mode predictor.
INFO:tensorflow:Loading model.


2026-01-13 02:30:57.398853: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:57.399250: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:57.399543: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:57.400064: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:30:57.400075: I tensorflow/core/co

INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


In [40]:
P_average

0.3753854337379487

### SoftRecall

In [41]:
softr_data = merged_df.sort_values(by=['category', 'product_name', 'user_id', 'key_point_given'])
df_compare_recall = softp_data[['key_point', 'key_point_given']].rename(columns={'key_point_given': 'candidate', 'key_point': 'reference'})
df_compare_recall

,reference,candidate
1,The Gillette Mach 3 Razor provides a close and...,Swapping blades is easy with the single-point ...
1,The Gillette Mach 3 Razor provides a close and...,The razor's design prevents hair from clogging...
1,The Gillette Mach 3 Razor provides a close and...,The swivel head and spring mechanism allow the...
1,The Gillette Mach 3 Razor provides a close and...,The razor causes fewer nicks and less irritati...
1,The Gillette Mach 3 Razor provides a close and...,The three-blade system glides effortlessly for...
...,...,...
94,The iconic Duomo is a must-see attraction in M...,The market near Piazzale Piemonte and Piazza S...
94,The iconic Duomo is a must-see attraction in M...,Visiting during the San Ambrogio festival adds...
94,The iconic Duomo is a must-see attraction in M...,Favoring trams over buses during peak times im...
94,The iconic Duomo is a must-see attraction in M...,Choosing markets over touristy eateries provid...


In [42]:
candidates = df_compare_recall['candidate']
references = df_compare_recall['reference']

In [43]:
import tensorflow as tf

In [44]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [45]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 1845405211133384840
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 19436404736
locality {
  bus_id: 1
  links {
  }
}
incarnation: 4125331324553299620
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


2026-01-13 02:36:31.413217: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:36:31.413870: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:36:31.414472: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:36:31.415003: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-13 02:36:31.415017: I tensorflow/core/co

In [46]:
result = calculatingScore(references, candidates)
df_compare_recall["BLEURT Score"] = result

#After calculating the semantic quality of all candidates and reference pairs, the one with the highest score is selected as the correct pair. 
df_bestkp_pair_recall = df_compare_recall.loc[df_compare_recall.groupby(["candidate"])["BLEURT Score"].idxmax()]

#take average of all best scores as the soft precision score.
R_average = df_bestkp_pair_recall["BLEURT Score"].mean()

INFO:tensorflow:Reading checkpoint /mnt/d/Desktop/PHD READING/HELPFULSUMM/evaluation/KeyPoint-Analysis/KPG/BLEURT-20.


INFO:tensorflow:Reading checkpoint /mnt/d/Desktop/PHD READING/HELPFULSUMM/evaluation/KeyPoint-Analysis/KPG/BLEURT-20.


INFO:tensorflow:Config file found, reading.


INFO:tensorflow:Config file found, reading.


INFO:tensorflow:Will load checkpoint BLEURT-20


...

INFO:tensorflow:Creating Eager Mode predictor.


INFO:tensorflow:Loading model.


INFO:tensorflow:Loading model.


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


In [47]:
R_average

0.3687534159128268

### F1

In [48]:
result = softF1(P_average, R_average)

In [49]:
result

0.372039871429235

## ROUGE

In [50]:
gt_gold_kp = merged_df

In [51]:
gt_gold_kp

,key_point,personalized_summaries,category,product_name,user_id,product_reviews,hist_vote_written,filtered_hist_vote_written,my_category,key_point_given
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,Head & Shoulders Normal Hair Shampoo contains ...
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,The shampoo keeps your scalp clear and flake-f...
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,Some users have found it doubles as a body was...
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,Head & Shoulders offers formulas tailored to d...
0,Head & Shoulders Normal Hair Shampoo is effect...,Here is a personalized summary of product A (H...,Beauty,Head & Shoulders Normal Hair Shampoo,3680,[ Valium is something that probably everyon...,[ i love alberto balsams esspecially the su...,[ i love alberto balsams esspecially the su...,1,The shampoo is effective for dandruff control ...
...,...,...,...,...,...,...,...,...,...,...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,Classical concerts and opera performances are ...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,Gellert Bath House offers stunning architectur...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,Nightlife options like Bahnhof nightclub have ...
95,Visiting Budapest leaves you with unforgettabl...,Here is a personalized summary of Budapest tai...,Travel,Budapest (Hungary),5116809,[ With Easyjet now offering budget flights ...,"[ The Lenin Mausoleum, a red and black gran...",[],1,"Budapest balances rich history, cultural delig..."


In [52]:
predictions, references = [], []
for topic in sorted(gt_gold_kp['product_name'].unique()):
    kps = gt_gold_kp.loc[(gt_gold_kp['product_name']==topic), 'key_point'].unique().tolist()
    gold_kps = gt_gold_kp.loc[(gt_gold_kp['product_name']==topic), 'key_point_given'].unique().tolist()
    if len(kps) > 0 and len(gold_kps) > 0:
        predictions.append(kps)
        references.append(gold_kps)

In [53]:
compute_rouge(predictions, references)

Rouge 1: 0.152
Rouge 2: 0.027
Rouge L: 0.138
